# Phase 13: Verification Completeness, Sentiment Intelligence, System Resilience — Replication Notebook

**Project:** HiFi — High-Fidelity Financial Intelligence  
**Phase:** 13  
**David sections:** SS9.4, SS10.3, SS10.4, SS14.4, SS8.7  
**Last updated:** 2026-06-15  

---

## Purpose

Frozen replication notebook for Phase 13. All sections load from JSON artifacts;
no LLM calls are made. Target runtime: **< 30 seconds**.

## Sections

| # | Section | Artifact | OQ |
|---|---|---|---|
| 1 | E0: Verification Baselines (Risk, Macro, Sentiment) | `phase13_verification_baseline.json` | — |
| 2 | E1: Sentiment Corpus Gate | `phase13_sentiment_corpus.json` | OQ-S01 |
| 3 | E2: Multi-Round Debate Herding | `phase13_debate_multiround.json` | OQ-D04 |
| 4 | E4: Agent Memory Influence | `phase13_memory_eval.json` | OQ-M03 |
| 5 | E5: Drift Monitor Calibration | `phase13_drift_calibration.json` | OQ-DR01 |
| 6 | E6: Scenario Alignment (Dataset Family F) | `data/scenarios/scenario_summary.json` | — |

In [1]:
"""Setup: path resolution and imports."""
import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings('ignore')

_nb = Path('.').resolve()
ROOT = _nb.parent if (_nb.parent / 'src').exists() else _nb
sys.path.insert(0, str(ROOT / 'src'))

BASELINE_DIR   = ROOT / 'tests' / 'fixtures' / 'baseline'
SCENARIOS_DIR  = ROOT / 'data' / 'scenarios'

VERIF_JSON     = BASELINE_DIR / 'phase13_verification_baseline.json'
CORPUS_JSON    = BASELINE_DIR / 'phase13_sentiment_corpus.json'
DEBATE_JSON    = BASELINE_DIR / 'phase13_debate_multiround.json'
MEMORY_JSON    = BASELINE_DIR / 'phase13_memory_eval.json'
DRIFT_JSON     = BASELINE_DIR / 'phase13_drift_calibration.json'
SCENARIO_JSON  = SCENARIOS_DIR / 'scenario_summary.json'

artifacts = {
    'verification_baseline': VERIF_JSON,
    'sentiment_corpus':      CORPUS_JSON,
    'debate_multiround':     DEBATE_JSON,
    'memory_eval':           MEMORY_JSON,
    'drift_calibration':     DRIFT_JSON,
    'scenario_summary':      SCENARIO_JSON,
}

print(f'ROOT: {ROOT}')
print()
for name, path in artifacts.items():
    status = 'OK' if path.exists() else 'PENDING'
    print(f'  [{status:7s}] {name}: {path.name}')

ROOT: /Users/alberto/Documents/projects/HiFi

  [OK     ] verification_baseline: phase13_verification_baseline.json
  [OK     ] sentiment_corpus: phase13_sentiment_corpus.json
  [OK     ] debate_multiround: phase13_debate_multiround.json
  [OK     ] memory_eval: phase13_memory_eval.json
  [OK     ] drift_calibration: phase13_drift_calibration.json
  [OK     ] scenario_summary: scenario_summary.json


---
## Section 1: E0 — Verification Layer Baselines (Risk, Macro, Sentiment)

Phase 13 E0 extends `verify_agent()` to all 5 voting agents. The baseline was
collected on 2023-03-31 using the fixture: `tests/fixtures/baseline/phase13_verification_baseline.json`.

Key results (DJ-072):
- **Risk:** HR=0.000, GR=1.000 — max_drawdown verified; alias_coverage=38.9% (vol aliases unresolvable).
- **Macro:** HR=0.000, GR=0.000, n_claims=0-1 — FRED data absent/sparse for 2023-03-31.
- **Sentiment (DJ-087, verbatim Rule 5):** mean_SGR=0.667 (4/6 grounded).
  AAPL SGR=0.000 (context is 8-K boilerplate), JPM SGR=1.000, XOM SGR=1.000.

In [2]:
if VERIF_JSON.exists():
    verif = json.loads(VERIF_JSON.read_text())
    _placeholder = False
else:
    # Placeholder matching actual baseline values
    verif = {
        'risk': {'hr': 0.000, 'gr': 1.000, 'alias_coverage': 0.389},
        'macro': {'hr': 0.000, 'gr': 0.000, 'n_claims_mean': 0.5},
        'sentiment': {
            'mean_sgr': 0.667,
            'per_ticker': {'AAPL': 0.000, 'JPM': 1.000, 'XOM': 1.000},
        },
    }
    _placeholder = True
    print('[PLACEHOLDER using actual baseline values — run scripts/run_phase13_e0_baseline.py]')

# Extract
risk = verif.get('risk', {})
macro = verif.get('macro', {})
sent = verif.get('sentiment', {})

print('E0 Verification Baselines (2023-03-31)' + (' [PLACEHOLDER]' if _placeholder else ''))
print('=' * 50)
print(f"Risk  Agent — HR: {risk.get('hr', 'N/A'):.3f}  GR: {risk.get('gr', 'N/A'):.3f}")
if 'alias_coverage' in risk:
    print(f"        alias_coverage: {risk['alias_coverage']:.3f} (vol aliases unresolvable)")
print()
print(f"Macro Agent — HR: {macro.get('hr', 'N/A'):.3f}  GR: {macro.get('gr', 'N/A'):.3f}")
if 'n_claims_mean' in macro:
    print(f"        n_claims (mean): {macro['n_claims_mean']} (FRED data sparse)")
print()
print(f"Sentiment — mean SGR: {sent.get('mean_sgr', sent.get('mean_sgr', 'N/A')):.3f}")
per_ticker = sent.get('per_ticker', {})
for ticker, sgr in per_ticker.items():
    note = ' (8-K boilerplate, no quotable signals)' if ticker == 'AAPL' and sgr == 0.0 else ''
    print(f"  {ticker}: SGR={sgr:.3f}{note}")

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# Risk
ax = axes[0]
ax.bar(['HR', 'GR', 'Alias\nCoverage'],
       [risk.get('hr', 0), risk.get('gr', 1), risk.get('alias_coverage', 0.389)],
       color=['#e74c3c', '#2ecc71', '#4a90e2'])
ax.set_ylim(0, 1.2)
ax.set_title('Risk Agent Verification')
ax.set_ylabel('Rate')

# Macro
ax = axes[1]
ax.bar(['HR', 'GR'], [macro.get('hr', 0), macro.get('gr', 0)],
       color=['#e74c3c', '#e74c3c'])
ax.set_ylim(0, 1.2)
ax.set_title('Macro Agent Verification\n(FRED data sparse)')
ax.set_ylabel('Rate')

# Sentiment SGR per ticker
ax = axes[2]
tickers = list(per_ticker.keys()) if per_ticker else ['AAPL', 'JPM', 'XOM']
sgrs    = [per_ticker.get(t, 0) for t in tickers]
ax.bar(tickers, sgrs, color=['#e74c3c', '#2ecc71', '#2ecc71'])
ax.axhline(y=0.720, color='orange', linewidth=1.5, linestyle='--', label='FT gate (0.720)')
ax.set_ylim(0, 1.3)
ax.set_title(f'Sentiment SGR per Ticker\n(mean={sent.get("mean_sgr", 0.667):.3f}, DJ-087)')
ax.set_ylabel('SGR')
ax.legend(fontsize=8)

plt.suptitle('Phase 13 E0: Verification Layer Baselines (2023-03-31)', fontsize=12)
plt.tight_layout()
plt.show()

E0 Verification Baselines (2023-03-31)


ValueError: Unknown format code 'f' for object of type 'str'

---
## Section 2: E1 — Sentiment Corpus Gate (OQ-S01)

Phase 13 E1 gated Sentiment fine-tuning on OQ-S01: is the corpus sufficient
(>= 100 Sell examples required for class balance)?

**ANSWERED NEGATIVE (E1-T1 ABORT).** The Phase 13 fixture-only corpus contains
0 Sell examples (3-ticker fixture, EDGAR not ingested). Sentiment FT deferred to Phase 14.

**DJ-088 implication:** Phase 14 must ingest EDGAR filings for >= 10 tickers before
Sentiment fine-tuning can proceed.

In [ ]:
if CORPUS_JSON.exists():
    corpus = json.loads(CORPUS_JSON.read_text())
    _placeholder = False
else:
    corpus = {
        'n_total': 0,
        'class_counts': {'Buy': 0, 'Hold': 0, 'Sell': 0},
        'oq_s01': 'NEGATIVE — 0 Sell examples; corpus insufficient for FT',
        'gate_passed': False,
    }
    _placeholder = True
    print('[PLACEHOLDER — actual result: 0 Sell examples, gate failed]')

class_counts = corpus.get('class_counts', {})
gate_passed = corpus.get('gate_passed', False)
oq_s01 = corpus.get('oq_s01', 'N/A')

print('E1: Sentiment Corpus Gate (OQ-S01)' + (' [PLACEHOLDER]' if _placeholder else ''))
print('=' * 50)
print(f"Total examples : {corpus.get('n_total', 0)}")
for label, count in class_counts.items():
    bar = '█' * count if count < 100 else '█' * 50 + f'+{count-50}'
    print(f"  {label:5s}: {count:4d}  {bar}")
print(f"Gate passed    : {gate_passed}  (threshold: >= 100 Sell examples)")
print(f"OQ-S01         : {oq_s01}")
print()
print('Action: Sentiment FT deferred to Phase 14.')
print('Phase 14 prerequisite: EDGAR ingestion for >= 10 tickers.')

---
## Section 3: E2 — Multi-Round Debate Herding (OQ-D04)

Tests whether a second debate round reduces herding vs. one round (Phase 12 condition C baseline).

**Design:** 5 dates × 3 tickers = 15 runs at max_rounds=2.  
**Baseline (Phase 12 condition C):** mean herding = 0.950.  
**Pre-registered hypothesis:** NEGLIGIBLE — herding is determined by architectural
diversity, not round count. A second round may reinforce the majority view (anchoring).

**Criterion:** |Δ| < 0.05 = NEGLIGIBLE; Δ < -0.05 = POSITIVE; Δ > +0.05 = NEGATIVE (anchoring).

In [ ]:
if DEBATE_JSON.exists():
    debate = json.loads(DEBATE_JSON.read_text())
    _placeholder = False
else:
    debate = {
        'baseline_1round_herding_condition_c': 0.950,
        'mean_herding_2round': None,
        'delta_2round_minus_1round': None,
        'oq_d04': 'PENDING',
        'per_run_results': [],
        'metadata': {'n_runs_completed': 0, 'n_runs_planned': 15},
    }
    _placeholder = True
    print('[PENDING] Run: uv run python scripts/run_phase13_debate_eval.py')

h1 = debate.get('baseline_1round_herding_condition_c', 0.950)
h2 = debate.get('mean_herding_2round')
delta = debate.get('delta_2round_minus_1round')
oq_d04 = debate.get('oq_d04', 'PENDING')
meta = debate.get('metadata', {})

print('E2: Multi-Round Debate Herding (OQ-D04)' + (' [PENDING]' if _placeholder else ''))
print('=' * 55)
print(f"Runs completed   : {meta.get('n_runs_completed', 0)}/{meta.get('n_runs_planned', 15)}")
print(f"Herding 1-round  : {h1:.4f}  (Phase 12 condition C baseline)")
if h2 is not None:
    print(f"Herding 2-round  : {h2:.4f}")
    print(f"Delta (2r - 1r)  : {delta:+.4f}")
    if abs(delta) < 0.05:
        verdict = 'NEGLIGIBLE — 2nd round does not change herding meaningfully'
    elif delta < -0.05:
        verdict = 'POSITIVE — 2nd round reduces herding'
    else:
        verdict = 'NEGATIVE — 2nd round increases herding (anchoring)'
    print(f"Verdict          : {verdict}")
else:
    print(f"Herding 2-round  : PENDING")
print(f"OQ-D04           : {oq_d04}")

# Visualise if data available
per_run = [r for r in debate.get('per_run_results', []) if 'herding_coefficient_2round' in r]
if per_run:
    fig, ax = plt.subplots(figsize=(10, 4))
    herding_vals = [r['herding_coefficient_2round'] for r in per_run]
    labels = [f"{r['ticker']}\n{r['as_of_date'][2:7]}" for r in per_run]
    x = np.arange(len(herding_vals))
    ax.bar(x, herding_vals, color='#4a90e2', alpha=0.85, label='2-round herding')
    ax.axhline(y=h1, color='red', linewidth=1.5, linestyle='--', label=f'1-round baseline ({h1:.3f})')
    if h2 is not None:
        ax.axhline(y=h2, color='green', linewidth=1.5, linestyle='-', label=f'2-round mean ({h2:.3f})')
    ax.set_ylim(0, 1.2)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel('Herding Coefficient')
    ax.set_title('Per-Run Herding Coefficient: 1-Round vs 2-Round Debate (OQ-D04)')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('(Chart available after eval completes)')

---
## Section 4: E4 — Agent Memory Influence (OQ-M03)

Tests whether in-context memory prefixes (last 3 decisions per agent per ticker)
change agent decisions vs. a no-memory baseline.

**Design:** 10 dates × 3 tickers = 30 pairs. Each pair run twice: (a) no memory,
(b) memory_prefixes populated from AgentMemoryStore with 3 synthetic prior records
(alternating Buy/Hold/Sell, with outcome metadata).  
**Agents:** fundamental + technical (matches Phase 12 factorial).  
**Metric:** Fraction of pairs where ≥1 agent changed decision.  
**Pre-registered hypothesis:** YES (weakly) — memory prefix creates anchoring bias
shifting ≥10% of decisions.

**Criterion:** changed_fraction >= 0.10 = YES; < 0.10 = NEGLIGIBLE.

In [ ]:
if MEMORY_JSON.exists():
    mem = json.loads(MEMORY_JSON.read_text())
    _placeholder = False
else:
    mem = {
        'n_pairs_changed': None,
        'changed_fraction': None,
        'oq_m03': 'PENDING',
        'metadata': {'n_pairs_planned': 30, 'n_pairs_completed': 0},
        'per_pair_results': [],
    }
    _placeholder = True
    print('[PENDING] Run: uv run python scripts/run_phase13_memory_eval.py')

frac = mem.get('changed_fraction')
n_changed = mem.get('n_pairs_changed')
oq_m03 = mem.get('oq_m03', 'PENDING')
meta = mem.get('metadata', {})

print('E4: Agent Memory Influence (OQ-M03)' + (' [PENDING]' if _placeholder else ''))
print('=' * 55)
print(f"Pairs completed    : {meta.get('n_pairs_completed', 0)}/{meta.get('n_pairs_planned', 30)}")
if frac is not None:
    print(f"Pairs changed (≥1) : {n_changed}/{meta.get('n_pairs_completed', 0)}")
    print(f"Changed fraction   : {frac:.3f}")
    print(f"Criterion          : >= 0.10 = YES; < 0.10 = NEGLIGIBLE")
else:
    print(f"Changed fraction   : PENDING")
print(f"OQ-M03             : {oq_m03}")

# Per-pair heatmap if data available
valid_pairs = [r for r in mem.get('per_pair_results', []) if 'any_changed' in r]
if valid_pairs:
    tickers = ['AAPL', 'JPM', 'XOM']
    dates_set = sorted(set(r['as_of_date'] for r in valid_pairs))
    matrix = np.zeros((len(tickers), len(dates_set)))
    for r in valid_pairs:
        ti = tickers.index(r['ticker']) if r['ticker'] in tickers else -1
        di = dates_set.index(r['as_of_date']) if r['as_of_date'] in dates_set else -1
        if ti >= 0 and di >= 0:
            matrix[ti, di] = 1.0 if r['any_changed'] else 0.0

    fig, ax = plt.subplots(figsize=(12, 3))
    im = ax.imshow(matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
    ax.set_yticks(range(len(tickers)))
    ax.set_yticklabels(tickers)
    ax.set_xticks(range(len(dates_set)))
    ax.set_xticklabels([d[2:7] for d in dates_set], rotation=30, fontsize=8)
    ax.set_title('Memory Influence Heatmap (Green=changed, Red=unchanged)')
    plt.colorbar(im, ax=ax, shrink=0.8, label='Changed (1) / Unchanged (0)')
    plt.tight_layout()
    plt.show()
else:
    print('(Heatmap available after eval completes)')

---
## Section 5: E5 — Drift Monitor Calibration (OQ-DR01)

Three drift monitors were calibrated against the 2022 rate-shock regime change.

**Monitors:**
- **KS test** — realized_vol + RSI: 2020-2021 vs 2022-2023 windows.
- **Chi-squared** — momentum proxy decisions: distribution shift in discrete votes.
- **CUSUM** — fraction of tickers below 50-day MA (2021 in-control baseline).

**OQ-DR01:** Do all three monitors detect the 2022 regime change?  
**Pre-registered hypothesis:** YES — 2022 (FFR 0→4.25%, CPI 8.5%) is the most
severe macro regime shift in the evaluation window.

In [ ]:
if DRIFT_JSON.exists():
    drift = json.loads(DRIFT_JSON.read_text())
    _placeholder = False
else:
    # Fill with actual baseline values from E5-T5
    drift = {
        'ks_test': {'statistic': None, 'p_value': 0.000, 'alert': True},
        'chi_squared': {'statistic': None, 'p_value': 0.000, 'alert': True},
        'cusum': {'C_k': 48.57, 'threshold': 0.534, 'alert': True},
        'oq_dr01': 'YES — all three monitors detect 2022 rate-shock regime change',
    }
    _placeholder = True
    print('[PLACEHOLDER using actual E5-T5 calibration values]')

ks   = drift.get('ks_test', {})
chi  = drift.get('chi_squared', {})
cus  = drift.get('cusum', {})
oq_dr01 = drift.get('oq_dr01', 'N/A')

print('E5: Drift Monitor Calibration (OQ-DR01)' + (' [PLACEHOLDER]' if _placeholder else ''))
print('=' * 55)
print(f"KS test (realized_vol + RSI):")
print(f"  p-value = {ks.get('p_value', 'N/A')}  alert = {ks.get('alert', 'N/A')}")
print()
print(f"Chi-squared (momentum proxy decisions):")
print(f"  p-value = {chi.get('p_value', 'N/A')}  alert = {chi.get('alert', 'N/A')}")
print()
print(f"CUSUM (frac tickers below 50d MA):")
print(f"  C_k = {cus.get('C_k', 'N/A')}  threshold = {cus.get('threshold', 'N/A')}  alert = {cus.get('alert', 'N/A')}")
if cus.get('C_k') and cus.get('threshold'):
    ratio = cus['C_k'] / cus['threshold']
    print(f"  C_k / threshold = {ratio:.1f}x")
print()
print(f"OQ-DR01: {oq_dr01}")

# Bar chart: monitor sensitivity
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# KS p-value
ax = axes[0]
p_ks = ks.get('p_value', 0.0)
ax.bar(['p-value'], [p_ks if p_ks is not None else 0], color='#e74c3c' if ks.get('alert') else '#4a90e2')
ax.axhline(y=0.05, color='orange', linewidth=1.5, linestyle='--', label='α=0.05')
ax.set_ylim(0, 0.15)
ax.set_title('KS Test (vol + RSI)\n2020-21 vs 2022-23')
ax.set_ylabel('p-value')
ax.legend(fontsize=8)
ax.text(0, (p_ks or 0) + 0.003, 'ALERT' if ks.get('alert') else 'NO ALERT', ha='center', fontsize=9)

# Chi-sq p-value
ax = axes[1]
p_chi = chi.get('p_value', 0.0)
ax.bar(['p-value'], [p_chi if p_chi is not None else 0], color='#e74c3c' if chi.get('alert') else '#4a90e2')
ax.axhline(y=0.05, color='orange', linewidth=1.5, linestyle='--', label='α=0.05')
ax.set_ylim(0, 0.15)
ax.set_title('Chi-Squared\n(momentum decisions)')
ax.set_ylabel('p-value')
ax.legend(fontsize=8)
ax.text(0, (p_chi or 0) + 0.003, 'ALERT' if chi.get('alert') else 'NO ALERT', ha='center', fontsize=9)

# CUSUM C_k vs threshold
ax = axes[2]
c_k = cus.get('C_k', 0) or 0
thresh = cus.get('threshold', 1) or 1
ax.bar(['C_k', 'Threshold'], [c_k, thresh],
       color=['#e74c3c' if cus.get('alert') else '#4a90e2', '#95a5a6'])
ax.set_title('CUSUM\n(frac < 50d MA)')
ax.set_ylabel('Value')
ax.text(0, c_k + 0.5, f'{c_k:.1f}', ha='center', fontsize=9)
ax.text(1, thresh + 0.5, f'{thresh:.3f}', ha='center', fontsize=9)

plt.suptitle('Phase 13 E5: Drift Monitor Calibration — 2022 Rate-Shock Detection', fontsize=11)
plt.tight_layout()
plt.show()

---
## Section 6: E6 — Scenario Alignment (Dataset Family F)

Seven historical stress-test scenarios (not generative synthetic — see DJ-078
methodological limitation) were evaluated against the ensemble.

**Scenarios:**
- **F-001/b/c** (2020-03-16): Black Monday II — COVID crash. Expected: Risk-Off or Sell.
- **F-002/b/c** (2022-03-31): Fed rate shock. Mixed: Risk-Off (AAPL), Hold (JPM), Buy (XOM).
- **F-003** (2023-02-02): AAPL earnings beat. Expected: Buy.

**Alignment:** exact match for Buy/Hold/Sell; Hold or Sell both satisfy Risk-Off.

In [ ]:
if SCENARIO_JSON.exists():
    scen_data = json.loads(SCENARIO_JSON.read_text())
    _placeholder = False
else:
    scen_data = {
        'alignment_rate': None,
        'n_aligned': None,
        'per_regime_alignment': {},
        'scenarios': [],
        'metadata': {'n_scenarios': 7, 'n_completed': 0},
    }
    _placeholder = True
    print('[PENDING] Run: uv run python scripts/run_phase13_scenarios.py')

align_rate = scen_data.get('alignment_rate')
n_aligned  = scen_data.get('n_aligned')
meta       = scen_data.get('metadata', {})
scenarios  = scen_data.get('scenarios', [])
per_regime = scen_data.get('per_regime_alignment', {})

print('E6: Scenario Alignment (Dataset Family F)' + (' [PENDING]' if _placeholder else ''))
print('=' * 60)
print(f"Scenarios completed : {meta.get('n_completed', 0)}/{meta.get('n_scenarios', 7)}")
if align_rate is not None:
    print(f"Overall alignment   : {n_aligned}/{meta.get('n_completed', 0)} = {align_rate:.3f}")
else:
    print(f"Overall alignment   : PENDING")
print()

# Per-scenario table
completed = [s for s in scenarios if 'error' not in s and 'aligned' in s]
if completed:
    print(f"{'Scenario':8s} {'Ticker':5s} {'Date':12s} {'Regime':15s} {'Expected':10s} {'Decision':10s} {'Aligned':8s}")
    print('-' * 80)
    for s in completed:
        print(
            f"{s.get('scenario_id',''):8s} {s.get('ticker',''):5s} "
            f"{s.get('as_of_date',''):12s} {s.get('regime',''):15s} "
            f"{s.get('expected_direction',''):10s} {s.get('collective_decision','N/A'):10s} "
            f"{'YES' if s.get('aligned') else 'NO':8s}"
        )
    print()

# Per-regime bar chart
if per_regime:
    regimes = list(per_regime.keys())
    rates   = [per_regime[r]['aligned'] / per_regime[r]['total'] for r in regimes]
    totals  = [per_regime[r]['total'] for r in regimes]

    fig, ax = plt.subplots(figsize=(8, 4))
    colours = ['#e74c3c' if r < 0.5 else '#2ecc71' if r >= 0.67 else '#e67e22' for r in rates]
    bars = ax.bar(regimes, rates, color=colours, alpha=0.85)
    ax.axhline(y=0.5, color='grey', linewidth=0.8, linestyle='--', label='50% alignment')
    ax.set_ylim(0, 1.2)
    ax.set_ylabel('Alignment rate')
    ax.set_title('Scenario Alignment per Regime (Dataset Family F)')
    for bar, rate, total in zip(bars, rates, totals):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.03,
                f'{rate:.2f}\n(n={total})', ha='center', fontsize=9)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('(Chart available after eval completes)')

---
## Section 7: Phase 13 Open Question Summary

Consolidated answers to all Phase 13 open questions.

In [ ]:
def _load(path):
    return json.loads(path.read_text()) if path.exists() else {}

verif_d  = _load(VERIF_JSON)
corpus_d = _load(CORPUS_JSON)
debate_d = _load(DEBATE_JSON)
memory_d = _load(MEMORY_JSON)
drift_d  = _load(DRIFT_JSON)

def _get(d, *keys, default='PENDING'):
    for k in keys:
        if isinstance(d, dict) and k in d:
            d = d[k]
        else:
            return default
    return d if d is not None else default

print('Phase 13 Open Question Resolution')
print('=' * 60)
print()

oqs = [
    ('OQ-S01',  'Sentiment corpus sufficient for FT (>= 100 Sell)?',
     _get(corpus_d, 'oq_s01')),
    ('OQ-D04',  'Does 2nd debate round reduce herding vs 1-round?',
     _get(debate_d, 'oq_d04')),
    ('OQ-M03',  'Does memory prefix change agent decisions?',
     _get(memory_d, 'oq_m03')),
    ('OQ-DR01', 'Do all 3 drift monitors detect 2022 regime change?',
     _get(drift_d, 'oq_dr01')),
]

for oq_id, question, answer in oqs:
    status = '[ANSWERED]' if answer != 'PENDING' else '[PENDING] '
    print(f'{status} {oq_id}: {question}')
    if answer != 'PENDING':
        # Truncate long answers
        ans_str = str(answer)
        print(f'          → {ans_str[:100] + "..." if len(ans_str) > 100 else ans_str}')
    print()

print('Architecture decisions in this phase: DJ-071 through DJ-087')
print('See plans/PHASE_13_CONTEXT.md for full DJ log.')